# FlatRAG vs. GRAG v2 —— 对比评测

对 `eval/questions.json` 里的 35 条问题依次跑两条 pipeline，给每个回答打分，
产出 `eval/comparision_plan.md` Part 3 里描述的表格和总结。

**运行第 1 段（批量跑批）之前须知**：本 notebook 会发起真实的 DeepSeek API 调用——
35 条问题 × 2 条 pipeline × 每条 pipeline 若干次 LLM 调用（agent loop + judge + 生成），
总量在几百次调用的量级。运行第 1 段之前，请确认已配置好 `DEEPSEEK_API_KEY`，并且愿意
为这次调用付费。

**RAGAS 说明**：第 2 段的 `score_with_ragas()` 用 `ragas 0.4.x` 的新版 collections API
（`ragas.metrics.collections`）实现，judge LLM 复用同一个 `deepseek-chat`（`temperature=0`），
embedding 用本地的 `BAAI/bge-small-en-v1.5`（和 FlatRAG 检索用的是同一个模型，本地跑、不占
额外 API 配额）。**这会显著增加第 2 段的调用量**：标准 6 指标里 Faithfulness/Answer
Relevancy 等单条指标内部就可能触发多次 LLM 调用，30 条标准题 × 6 个指标下来，第 2 段自己
就是大几十到上百次额外的 judge 调用，跑起来比第 1 段的批跑本身还慢，请在预算里把这部分也
算进去。如果当前环境没装 `ragas`，或者构建 judge LLM/embedding 失败（比如没配置
`DEEPSEEK_API_KEY`），`score_with_ragas()` 会自动降级为返回 `NaN`，不会导致 notebook 跑不通，
后面所有表格（top/bottom 案例、分歧表等）在真实分数和 `NaN` 两种情况下都能正常出表格。

## 环境准备

导入依赖、定位仓库根目录，把 `ui/` 和仓库根目录加入 `sys.path`，加载 `.env`（供后面 RAGAS judge LLM 读取 `DEEPSEEK_API_KEY`），并加载 35 条评测问题（`questions.json`）。

In [12]:
from __future__ import annotations

import asyncio
import json
import math
import os
import sys
import warnings
from datetime import datetime
from pathlib import Path

import pandas as pd


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "eval" / "questions.json").exists() and (candidate / "ui" / "adapters").exists():
            return candidate
    raise RuntimeError(f"Could not locate repo root from {start}")


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
for _p in (REPO_ROOT, REPO_ROOT / "ui"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# The adapters also load these internally, but Section 2's RAGAS judge LLM needs
# DEEPSEEK_API_KEY in os.environ too, so load explicitly rather than depend on that
# side effect of adapter instantiation running first.
from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")
load_dotenv(REPO_ROOT / "pipeline_flatrag" / ".env")

from eval.comparison_lib import RUNS_DIR, _load_questions, _question_meta, _write_run_log
from adapters.pipeline_flatrag_adapter import FlatRAGAdapter
from adapters.pipeline_grag_v2_adapter import GRAGv2Adapter

REPORTS_DIR = REPO_ROOT / "eval" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_IDS = ["pipeline_flatrag", "pipeline_grag_v2"]
STANDARD_TYPES = {"single_fact", "multi_fact", "table", "accordion", "cross_page", "comparative"}
EDGE_TYPES = {"out_of_scope", "missing_info"}
RAGAS_METRIC_COLUMNS = [
    "context_precision", "context_recall", "context_relevancy",
    "faithfulness", "answer_relevancy", "answer_correctness",
]

# comparision_plan.md Part 1's 5 major categories -- aggregation tables group by these
# instead of the 8 raw `type` values, so table/accordion and cross_page/comparative (and,
# for the edge type, out_of_scope/missing_info) each roll up into one row.
TYPE_CATEGORY_LABELS = {
    "single_fact": "单一事实",
    "multi_fact": "多部分问题",
    "table": "结构化页面",
    "accordion": "结构化页面",
    "cross_page": "跨页综合",
    "comparative": "跨页综合",
    "out_of_scope": "无答案/边界问题",
    "missing_info": "无答案/边界问题",
}
STANDARD_CATEGORY_ORDER = ["单一事实", "多部分问题", "结构化页面", "跨页综合"]
EDGE_CATEGORY_ORDER = ["无答案/边界问题"]
PIPELINE_LABELS = {"pipeline_flatrag": "flatrag", "pipeline_grag_v2": "grag"}
SCORE_ROUND = 2

questions = _load_questions()
assert len(questions) == 35, f"expected 35 questions in questions.json, found {len(questions)}"
questions_by_id = {q["id"]: q for q in questions}
print(f"Loaded {len(questions)} questions from {REPO_ROOT / 'eval' / 'questions.json'}")

Loaded 35 questions from C:\Users\ss363\Desktop\Archive\ads_rag_chatbot\eval\questions.json


实例化两条 pipeline 的 adapter（只加载一次 embedding 模型和索引，供后面全部 35 条问题复用，避免每条问题都重新加载一遍）。

In [2]:
# Loads the embedding model / Chroma collections / KG index once and reuses
# these adapter instances for all 35 questions, instead of re-instantiating
# (and re-loading indices) per question.
print("Instantiating adapters (loads embedding models / indices once)...")
flat_adapter = FlatRAGAdapter()
grag_adapter = GRAGv2Adapter()
ADAPTERS = {"pipeline_flatrag": flat_adapter, "pipeline_grag_v2": grag_adapter}
print("Adapters ready.")

Instantiating adapters (loads embedding models / indices once)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Adapters ready.


## 第 1 段 — 批量跑一遍全部问题，存到本地

对每条问题依次跑两条 pipeline，每条问题通过 `eval.comparison_lib._write_run_log`
写一条 log 到 `eval/comparison_runs/`（和 Streamlit UI 写的是同一套 schema）。
支持断点续跑：如果 kernel 中途挂了，直接重新跑这个 cell 即可——它会自动续跑最近一次
未跑完的 `run_batch_id`，不会从头重来、重复为已经跑完的部分付费。

定义批次进度扫描、续跑判断相关的辅助函数，并据此确定本次要用的 `run_batch_id`——自动续跑上一次未完成的批次，或者在上一批已经跑完时开启一个新批次。

In [3]:
# Returns the (question_id, pipeline_id) pairs already successfully logged under batch_id.
def _load_batch_progress(batch_id: str) -> set[tuple[str, str]]:
    done: set[tuple[str, str]] = set()
    for path in RUNS_DIR.glob("*.json"):
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError):
            continue
        if payload.get("run_batch_id") != batch_id:
            continue
        qid = payload.get("question_id")
        if not qid:
            continue
        for pipeline_id, result in (payload.get("results") or {}).items():
            if result and result.get("error") is None:
                done.add((qid, pipeline_id))
    return done


def _discover_latest_batch_id() -> str | None:
    ids: set[str] = set()
    for path in RUNS_DIR.glob("*.json"):
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError):
            continue
        bid = payload.get("run_batch_id")
        if bid:
            ids.add(bid)
    return max(ids) if ids else None  # "%Y%m%dT%H%M%S" sorts correctly as a string


def _is_batch_complete(batch_id: str) -> bool:
    return len(_load_batch_progress(batch_id)) >= len(questions) * len(PIPELINE_IDS)


# Set to a specific run_batch_id string (see filenames' "run_batch_id" field)
# to force-resume a particular batch. Leave as None to auto-resume the most
# recent incomplete batch, or start a fresh one if the last batch finished.
FORCE_RESUME_RUN_BATCH_ID: str | None = None

if FORCE_RESUME_RUN_BATCH_ID:
    run_batch_id = FORCE_RESUME_RUN_BATCH_ID
else:
    _latest = _discover_latest_batch_id()
    if _latest and not _is_batch_complete(_latest):
        run_batch_id = _latest
        print(f"Resuming incomplete batch: {run_batch_id}")
    else:
        run_batch_id = datetime.now().strftime("%Y%m%dT%H%M%S")
        print(f"Starting new batch: {run_batch_id}")

already_done = _load_batch_progress(run_batch_id)
print(f"{len(already_done)}/{len(questions) * len(PIPELINE_IDS)} (question_id, pipeline) pairs already completed for this batch.")

Starting new batch: 20260804T013937
0/70 (question_id, pipeline) pairs already completed for this batch.


核心批跑循环：对每条还没跑完的 (问题, pipeline) 组合调用对应 adapter；单条调用失败会被记录下来但不会中断整批，每条问题两条 pipeline 都跑完后写入 `comparison_runs` 日志。

In [4]:
def _error_result(pipeline_id: str, exc: Exception) -> dict:
    return {
        "pipeline_id": pipeline_id,
        "display_name": pipeline_id,
        "answer": "",
        "sources": [],
        "steps": [],
        "evidence": [],
        "time_sec": 0.0,
        "low_confidence": True,
        "error": str(exc),
        "raw_debug": {},
    }


for _q in questions:
    qid = _q["id"]
    pending = [pid for pid in PIPELINE_IDS if (qid, pid) not in already_done]
    if not pending:
        continue

    q_meta = _question_meta(_q)
    results_for_question: dict[str, dict] = {}
    for pipeline_id in pending:
        adapter = ADAPTERS[pipeline_id]
        print(f"[run] {qid} / {pipeline_id} — {_q['question'][:70]}")
        try:
            result = adapter.run(_q["question"], history=[])
        except Exception as exc:  # noqa: BLE001 — one bad call must not kill the batch
            print(f"  ! {pipeline_id} raised {exc!r}")
            result = _error_result(pipeline_id, exc)
        results_for_question[pipeline_id] = result

    _write_run_log(
        _q["question"],
        results_for_question,
        question_id=qid,
        question_meta=q_meta,
        run_batch_id=run_batch_id,
    )

print("Batch run loop complete.")

[run] A01 / pipeline_flatrag — How many letters of recommendation do I need for the MS in Applied Dat
[run] A01 / pipeline_grag_v2 — How many letters of recommendation do I need for the MS in Applied Dat
[run] A04 / pipeline_flatrag — What application materials do I need to submit for the MS in Applied D
[run] A04 / pipeline_grag_v2 — What application materials do I need to submit for the MS in Applied D
[run] A05 / pipeline_flatrag — Which MS-ADS program options are visa-eligible for international stude
[run] A05 / pipeline_grag_v2 — Which MS-ADS program options are visa-eligible for international stude
[run] A06 / pipeline_flatrag — Under what circumstance must an MS-ADS applicant submit proof of Engli
[run] A06 / pipeline_grag_v2 — Under what circumstance must an MS-ADS applicant submit proof of Engli
[run] A07 / pipeline_flatrag — Which standardized test scores are optional for MS-ADS applicants?
[run] A07 / pipeline_grag_v2 — Which standardized test scores are optional for MS-ADS 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[run] E05 / pipeline_flatrag — Between UChicago and Stanford, which school's program is better for so
[run] E05 / pipeline_grag_v2 — Between UChicago and Stanford, which school's program is better for so
[run] E06 / pipeline_flatrag — What is the average class size for MS-ADS core courses?
[run] E06 / pipeline_grag_v2 — What is the average class size for MS-ADS core courses?
[run] F01 / pipeline_flatrag — Who is the director of the MS in Applied Data Science program at UChic
[run] F01 / pipeline_grag_v2 — Who is the director of the MS in Applied Data Science program at UChic
[run] F03 / pipeline_flatrag — Which courses does Arnab Bose teach in the MS-ADS program?
[run] F03 / pipeline_grag_v2 — Which courses does Arnab Bose teach in the MS-ADS program?
[run] F04 / pipeline_flatrag — Which MS-ADS instructors have worked at Google?
[run] F04 / pipeline_grag_v2 — Which MS-ADS instructors have worked at Google?
[run] S01 / pipeline_flatrag — How does the age distribution differ between the 

把本次 `run_batch_id` 下写入的全部日志文件重新读回内存，展平 `usage`/`metrics` 字段、从 `evidence` 里提取每条结果的 `retrieved_contexts`（RAGAS 打分要用），汇总成 `batch_results` 列表（第 2-4 段都直接用这份内存数据，不再重新扫盘），并校验这一批是否已经跑完整。

In [5]:
def _flatten_metrics(result: dict) -> dict:
    raw_debug = result.get("raw_debug") or {}
    usage = raw_debug.get("usage") or {}
    metrics = raw_debug.get("metrics") or {}
    # Both adapters' evidence items use the same "text" key for the retrieved passage body
    # (pipeline_flatrag chunk dicts and pipeline_grag_v2's EvidenceItem.to_dict() alike), so
    # this works uniformly across both pipelines without a pipeline_id branch.
    evidence = result.get("evidence") or []
    retrieved_contexts = [e.get("text", "") for e in evidence if e.get("text")]
    # loop_count/evidence_count are FlatRAG's key names for concepts GRAG v2's pre-existing
    # metrics dict names agent_turns/accepted_evidence_count instead -- fall back to those so
    # both pipelines land in the same flattened column rather than GRAG v2 silently reading
    # as NaN here (its data would otherwise never show up in the aggregation tables at all).
    loop_count = metrics.get("loop_count")
    if loop_count is None:
        loop_count = metrics.get("agent_turns")
    evidence_count = metrics.get("evidence_count")
    if evidence_count is None:
        evidence_count = metrics.get("accepted_evidence_count")
    return {
        "total_tokens": usage.get("total_tokens"),
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "llm_calls": usage.get("llm_calls"),
        "tool_calls": metrics.get("tool_calls"),
        "loop_count": loop_count,
        "evidence_count": evidence_count,
        "judge_calls": metrics.get("judge_calls"),
        "fallback_triggered": metrics.get("fallback_triggered"),
        "retrieved_contexts": retrieved_contexts,
    }


# Reads every log written under batch_id back into one flat in-memory list. Reading back
# from disk (rather than only using what this session just ran) means Section 2-4 see the
# complete batch even after a resume, and do their aggregation entirely off this one
# in-memory list -- no further disk scans.
#
# A resume can retry just one pipeline for a question that partially succeeded (e.g.
# FlatRAG succeeded, GRAG v2 errored), which writes a second, separate log file for that
# same question_id. Filenames are timestamp-prefixed so glob() sorts chronologically;
# keying by (question_id, pipeline_id) and letting later files overwrite earlier ones in
# that sorted iteration gives correct last-write-wins semantics instead of double-counting
# the same pipeline's result once per log file it appears in.
def _collect_batch_results(batch_id: str) -> list[dict]:
    latest: dict[tuple[str, str], dict] = {}
    for path in sorted(RUNS_DIR.glob("*.json")):
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError):
            continue
        if payload.get("run_batch_id") != batch_id:
            continue
        qid = payload.get("question_id")
        q = questions_by_id.get(qid)
        if q is None:
            continue
        for pipeline_id, result in (payload.get("results") or {}).items():
            result = result or {}
            latest[(qid, pipeline_id)] = {
                "question_id": qid,
                "question": q["question"],
                "type": q["type"],
                "difficulty": q["difficulty"],
                "category": q["category"],
                "reference_answer": q["reference_answer"],
                "pipeline_id": pipeline_id,
                "answer": result.get("answer", ""),
                "time_sec": result.get("time_sec"),
                "low_confidence": result.get("low_confidence"),
                "error": result.get("error"),
                "log_path": str(path),
                **_flatten_metrics(result),
            }
    return list(latest.values())


batch_results = _collect_batch_results(run_batch_id)
n_errors = sum(1 for r in batch_results if r["error"] is not None)
print(f"{len(batch_results)} total (question, pipeline) results in batch {run_batch_id}; {n_errors} had errors and will be excluded from scoring.")
assert len(batch_results) == len(questions) * len(PIPELINE_IDS), "batch is incomplete — re-run the previous cell to finish it"

70 total (question, pipeline) results in batch 20260804T013937; 0 had errors and will be excluded from scoring.


## 第 2 段 — 用 RAGAS 给每条 (question, pipeline) 结果打分

30 条标准题型（`single_fact`/`multi_fact`/`table`/`accordion`/`cross_page`/`comparative`）
走标准 6 个 RAGAS 指标；5 条边界题型（`out_of_scope`/`missing_info`）没有正向证据可检索，
Context Precision/Recall 在没有 gold context 时没有意义，所以改成确定性的拒答话术分类，
不需要额外 LLM 调用。

定义边界题（`out_of_scope`/`missing_info`）拒答话术的确定性分类函数；构建 RAGAS 0.4.x 的 judge LLM（DeepSeek，`temperature=0`）和本地 embedding 模型；实现 `score_with_ragas`，对标准题真正跑 6 项 RAGAS 指标（没装 `ragas` 或构建 judge 失败时优雅降级为 `NaN`，不影响 notebook 跑通）。

In [6]:
# Both pipelines are prompted (pipeline_grag_v2/agent/prompts.py ANSWER_GENERATION_SYSTEM,
# pipeline_flatrag/src/qa/prompt_templates.py SYSTEM_PROMPT rules 1-2) to reply with one of
# these two exact boilerplate strings. Matching against them is exact and free, unlike RAGAS
# metrics which need positive gold context that edge questions don't have by design.
SCOPE_REFUSAL = "I can only answer questions about the UChicago MS in Applied Data Science program."
INSUFFICIENT_CONTEXT_REFUSAL = (
    "I couldn't find relevant information about this in the available program materials. "
    "I can only answer questions about the UChicago MS in Applied Data Science program."
)
# INSUFFICIENT_CONTEXT_REFUSAL ends with the exact text of SCOPE_REFUSAL, so it must be
# checked first — otherwise every missing_info refusal would also match as out_of_scope.


def _classify_refusal_kind(answer_text: str) -> str:
    text = (answer_text or "").strip()
    if INSUFFICIENT_CONTEXT_REFUSAL in text:
        return "missing_info_refusal"
    if SCOPE_REFUSAL in text:
        return "out_of_scope_refusal"
    return "no_refusal_detected"


# Returns one of 正确拒答/拒答但话术类型用错/没有拒答编了内容 for an edge-type answer.
def score_refusal_correctness(answer_text: str, expected_type: str) -> str:
    kind = _classify_refusal_kind(answer_text)
    if kind == "no_refusal_detected":
        return "fabricated"
    expected_kind = "out_of_scope_refusal" if expected_type == "out_of_scope" else "missing_info_refusal"
    return "correct_refusal" if kind == expected_kind else "wrong_refusal_type"


_RAGAS_JUDGE_MODEL = "deepseek-chat"
_RAGAS_EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"  # same model FlatRAG's own retriever uses
_ragas_metrics: "dict[str, object] | None" = None

try:
    import ragas  # noqa: F401
    from openai import AsyncOpenAI
    from ragas.embeddings import HuggingFaceEmbeddings
    from ragas.llms import llm_factory
    from ragas.metrics.collections import (
        AnswerCorrectness,
        AnswerRelevancy,
        ContextPrecision,
        ContextRecall,
        ContextRelevance,
        Faithfulness,
    )

    _RAGAS_AVAILABLE = True
except ImportError:
    _RAGAS_AVAILABLE = False
    warnings.warn(
        "ragas is not installed -- standard-type rows will get NaN for the 6 RAGAS metrics. "
        "`pip install ragas` to compute real scores."
    )

if _RAGAS_AVAILABLE:
    try:
        _ragas_api_key = os.environ.get("DEEPSEEK_API_KEY") or os.environ.get("OPENAI_API_KEY")
        if not _ragas_api_key:
            raise RuntimeError("DEEPSEEK_API_KEY/OPENAI_API_KEY not set")
        # ragas 0.4.x's collections-API metrics call agenerate() internally, which requires
        # an async-capable client -- a plain sync OpenAI() raises "Cannot use agenerate()
        # with a synchronous client."
        _ragas_client = AsyncOpenAI(api_key=_ragas_api_key, base_url="https://api.deepseek.com")
        _ragas_llm = llm_factory(model=_RAGAS_JUDGE_MODEL, provider="openai", client=_ragas_client, temperature=0)
        # use_api=False runs BAAI/bge-small-en-v1.5 locally via sentence-transformers, same
        # as pipeline_flatrag/src/qa/qa_pipeline.py _MODEL_NAME -- no extra API key needed.
        _ragas_embeddings = HuggingFaceEmbeddings(model=_RAGAS_EMBEDDING_MODEL, use_api=False)
        _ragas_metrics = {
            "faithfulness": Faithfulness(llm=_ragas_llm),
            "answer_relevancy": AnswerRelevancy(llm=_ragas_llm, embeddings=_ragas_embeddings),
            "answer_correctness": AnswerCorrectness(llm=_ragas_llm, embeddings=_ragas_embeddings),
            "context_precision": ContextPrecision(llm=_ragas_llm),
            "context_recall": ContextRecall(llm=_ragas_llm),
            "context_relevancy": ContextRelevance(llm=_ragas_llm),
        }
    except Exception as exc:  # noqa: BLE001 — missing key / network issue must not block the notebook
        _RAGAS_AVAILABLE = False
        warnings.warn(f"Could not build RAGAS judge LLM/embeddings, falling back to NaN scores: {exc!r}")


def _run_async(coro):
    # metric.score() (the sync wrapper ragas ships) internally does asyncio.run(), which
    # raises inside an already-running event loop -- and modern Jupyter/ipykernel does run
    # one. Calling ascore() directly and, only when a loop is already running, executing it
    # on a dedicated thread's own loop instead works identically in a plain script and here.
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    import concurrent.futures
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _pool:
        return _pool.submit(lambda: asyncio.run(coro)).result()


def _ragas_kwargs(metric_name: str, row: dict) -> dict:
    user_input = row["question"]
    response = row["answer"]
    reference = row["reference_answer"]
    retrieved_contexts = row.get("retrieved_contexts") or []
    return {
        "faithfulness": dict(user_input=user_input, response=response, retrieved_contexts=retrieved_contexts),
        "answer_relevancy": dict(user_input=user_input, response=response),
        "answer_correctness": dict(user_input=user_input, response=response, reference=reference),
        "context_precision": dict(user_input=user_input, reference=reference, retrieved_contexts=retrieved_contexts),
        "context_recall": dict(user_input=user_input, retrieved_contexts=retrieved_contexts, reference=reference),
        "context_relevancy": dict(user_input=user_input, retrieved_contexts=retrieved_contexts),
    }[metric_name]


# Scores one (question, pipeline) result on the 6 standard RAGAS metrics using ragas 0.4.x's
# collections API. Each metric is independent: one metric erroring (timeout, malformed judge
# output, etc.) only NaNs that column, it doesn't lose the other 5 for this row.
def score_with_ragas(row: dict) -> dict:
    if not _RAGAS_AVAILABLE or _ragas_metrics is None:
        return {m: math.nan for m in RAGAS_METRIC_COLUMNS}

    scores = {}
    for name, metric in _ragas_metrics.items():
        try:
            result = _run_async(metric.ascore(**_ragas_kwargs(name, row)))
            scores[name] = result.value
        except Exception as exc:  # noqa: BLE001 — one metric failing must not sink the row
            warnings.warn(f"score_with_ragas: {name} failed for {row.get('question_id')}/{row.get('pipeline_id')}: {exc!r}")
            scores[name] = math.nan
    return scores

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

对 `batch_results` 逐行打分：标准题调用 `score_with_ragas` 占位函数，边界题调用拒答分类函数；打完分的结果汇总成打平的 `scored_df`，并落盘缓存成一份 CSV（避免重复触发打分调用）。

In [7]:
scored_rows = []
for row in batch_results:
    scored = dict(row)
    if row["error"] is not None:
        for m in RAGAS_METRIC_COLUMNS:
            scored[m] = math.nan
        scored["refusal_correct"] = None
        scored_rows.append(scored)
        continue

    q_type = row["type"]
    if q_type in EDGE_TYPES:
        scored["refusal_correct"] = score_refusal_correctness(row["answer"], q_type)
        for m in RAGAS_METRIC_COLUMNS:
            scored[m] = math.nan
    elif q_type in STANDARD_TYPES:
        scored.update(score_with_ragas(row))
        scored["refusal_correct"] = None
    else:
        raise ValueError(f"Unexpected question type {q_type!r} on {row['question_id']}")
    scored_rows.append(scored)

scored_df = pd.DataFrame(scored_rows)
cache_path = REPORTS_DIR / f"scored_results_{run_batch_id}.csv"
scored_df.to_csv(cache_path, index=False)
print(f"Scored table cached to {cache_path} ({len(scored_df)} rows).")

valid_df = scored_df[scored_df["error"].isna()].copy()
standard_df = valid_df[valid_df["type"].isin(STANDARD_TYPES)].copy()
edge_df = valid_df[valid_df["type"].isin(EDGE_TYPES)].copy()
print(f"{len(valid_df)} valid rows: {len(standard_df)} standard-type, {len(edge_df)} edge-type.")

C:\Users\ss363\AppData\Local\Temp\ipykernel_35508\533452931.py:126: UserWarning: score_with_ragas: faithfulness failed for C05/pipeline_flatrag: IncompleteOutputException('The output is incomplete due to a max_tokens length limit.')
  warnings.warn(f"score_with_ragas: {name} failed for {row.get('question_id')}/{row.get('pipeline_id')}: {exc!r}")
C:\Users\ss363\AppData\Local\Temp\ipykernel_35508\533452931.py:126: UserWarning: score_with_ragas: answer_correctness failed for C05/pipeline_flatrag: IncompleteOutputException('The output is incomplete due to a max_tokens length limit.')
  warnings.warn(f"score_with_ragas: {name} failed for {row.get('question_id')}/{row.get('pipeline_id')}: {exc!r}")
C:\Users\ss363\AppData\Local\Temp\ipykernel_35508\533452931.py:126: UserWarning: score_with_ragas: faithfulness failed for C05/pipeline_grag_v2: IncompleteOutputException('The output is incomplete due to a max_tokens length limit.')
  warnings.warn(f"score_with_ragas: {name} failed for {row.get('q

Scored table cached to C:\Users\ss363\Desktop\Archive\ads_rag_chatbot\eval\reports\scored_results_20260804T013937.csv (70 rows).
70 valid rows: 60 standard-type, 10 edge-type.


补跑修复：上一个 cell 里因 `max_tokens` 截断变成 `NaN` 的那几个 (问题, pipeline, 指标) 组合，用一个单独构建的、`max_tokens=4096` 的 judge LLM 实例重新打分（不动前面已经跑成功的部分、也不重跑整批）；打完后原地刷新 `scored_df`/`valid_df`/`standard_df`/`edge_df`，并覆盖写回缓存 CSV。

In [8]:
# Backfill: only re-scores the (row, metric) cells that are still NaN after the previous
# cell (typically ragas's default max_tokens=1024 truncating Faithfulness/Answer Correctness's
# structured statement breakdown on longer answers). Uses a separate, larger-max_tokens judge
# LLM instance so this doesn't just repeat the same failure; leaves already-scored cells alone.
_backfill_llm = llm_factory(model=_RAGAS_JUDGE_MODEL, provider="openai", client=_ragas_client, temperature=0, max_tokens=4096)
_backfill_metrics = {
    "faithfulness": Faithfulness(llm=_backfill_llm),
    "answer_relevancy": AnswerRelevancy(llm=_backfill_llm, embeddings=_ragas_embeddings),
    "answer_correctness": AnswerCorrectness(llm=_backfill_llm, embeddings=_ragas_embeddings),
    "context_precision": ContextPrecision(llm=_backfill_llm),
    "context_recall": ContextRecall(llm=_backfill_llm),
    "context_relevancy": ContextRelevance(llm=_backfill_llm),
}

_to_backfill = [
    (idx, m)
    for idx, row in scored_df.iterrows()
    if row["type"] in STANDARD_TYPES and pd.isna(row["error"])
    for m in RAGAS_METRIC_COLUMNS
    if pd.isna(row[m])
]
print(f"{len(_to_backfill)} (row, metric) cells to backfill.")

for idx, metric_name in _to_backfill:
    row = scored_df.loc[idx].to_dict()
    try:
        result = _run_async(_backfill_metrics[metric_name].ascore(**_ragas_kwargs(metric_name, row)))
        scored_df.loc[idx, metric_name] = result.value
        print(f"  fixed {row['question_id']}/{row['pipeline_id']}/{metric_name} -> {result.value}")
    except Exception as exc:  # noqa: BLE001 -- a still-failing cell just stays NaN
        warnings.warn(f"backfill: {metric_name} still failing for {row.get('question_id')}/{row.get('pipeline_id')}: {exc!r}")

scored_df.to_csv(cache_path, index=False)
print(f"Backfill done, re-cached to {cache_path}.")

valid_df = scored_df[scored_df["error"].isna()].copy()
standard_df = valid_df[valid_df["type"].isin(STANDARD_TYPES)].copy()
edge_df = valid_df[valid_df["type"].isin(EDGE_TYPES)].copy()
print(f"{len(valid_df)} valid rows: {len(standard_df)} standard-type, {len(edge_df)} edge-type.")

11 (row, metric) cells to backfill.
  fixed C05/pipeline_flatrag/faithfulness -> 1.0
  fixed C05/pipeline_flatrag/answer_correctness -> 0.5349434968887544
  fixed C05/pipeline_grag_v2/faithfulness -> 0.9411764705882353
  fixed C09/pipeline_grag_v2/faithfulness -> 0.631578947368421
  fixed C09/pipeline_grag_v2/answer_correctness -> 0.5558495324353868
  fixed C12/pipeline_flatrag/faithfulness -> 1.0
  fixed C12/pipeline_flatrag/answer_correctness -> 0.9735028734846634
  fixed C12/pipeline_grag_v2/faithfulness -> 1.0
  fixed C12/pipeline_grag_v2/answer_correctness -> 0.9805665813194937
  fixed T02/pipeline_grag_v2/faithfulness -> 1.0
  fixed X04/pipeline_flatrag/answer_correctness -> 0.9526048208995042
Backfill done, re-cached to C:\Users\ss363\Desktop\Archive\ads_rag_chatbot\eval\reports\scored_results_20260804T013937.csv.
70 valid rows: 60 standard-type, 10 edge-type.


## 第 3 段 — 按问题类型（`type`）统计两条 pipeline 的指标

复用第 2 段已经打分的 `valid_df`/`standard_df`/`edge_df`，按 comparision_plan.md 的 5 大类
（把 `table`/`accordion` 合成"结构化页面"、`cross_page`/`comparative` 合成"跨页综合"）分组，
统一输出 2 张表：① 质量指标表（标准 6 项 RAGAS 指标 + 边界题拒答正确率等，5 个大类各一行）；
② 成本指标表（平均延迟、平均 token 用量、平均 evidence 数，同样 5 个大类各一行）。下面紧接着
再用一张"GRAG v2 相对 FlatRAG 的增幅"表把这两张表都换算成两条 pipeline 的差值——RAGAS/拒答
类 0-1 量表的指标用百分点（pp）差，延迟/token 用相对变化的百分比，evidence 数用绝对差值。

按 comparision_plan.md 的 5 大类分组，输出两张表：质量指标表——标准题的 6 项 RAGAS 指标 + 边界题拒答正确率/错误话术率/编造率合并成一张表（标准列在边界行是 NaN，反之亦然），额外加一行“多信息结合”子总计（合并需要综合多条信息的 3 类：多部分问题/结构化页面/跨页综合），最下面再加一行覆盖全部 35 条问题的 `total` 行；成本指标表结构相同（平均耗时、平均 token 用量、平均 evidence 数）。两张表都按 pipeline 拆成 `flatrag`/`grag` 两列并在列名里标注单位，数值四舍五入到 2 位小数。

In [34]:
def _pivot_by_pipeline(df: pd.DataFrame, group_col: str, value_cols: list[str]) -> pd.DataFrame:
    agg = df.groupby([group_col, "pipeline_id"]).agg(
        n=("question_id", "count"),
        **{c: (c, "mean") for c in value_cols},
    ).reset_index()
    pivoted = agg.pivot(index=group_col, columns="pipeline_id")
    pivoted.columns = [f"{metric}__{PIPELINE_LABELS.get(pid, pid)}" for metric, pid in pivoted.columns]
    score_cols = [c for c in pivoted.columns if not c.startswith("n__")]
    pivoted[score_cols] = pivoted[score_cols].round(SCORE_ROUND)
    return pivoted.reset_index()


def _refusal_rate_table(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    rows = []
    for (group_val, pipeline_id), group in df.groupby([group_col, "pipeline_id"]):
        n = len(group)
        rows.append({
            group_col: group_val,
            "pipeline_id": pipeline_id,
            "n": n,
            "refusal_correct_rate": round((group["refusal_correct"] == "correct_refusal").sum() / n, SCORE_ROUND) if n else math.nan,
            "wrong_refusal_type_rate": round((group["refusal_correct"] == "wrong_refusal_type").sum() / n, SCORE_ROUND) if n else math.nan,
            "fabricated_rate": round((group["refusal_correct"] == "fabricated").sum() / n, SCORE_ROUND) if n else math.nan,
        })
    return pd.DataFrame(rows)


def _pivot_refusal_by_pipeline(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    long_df = _refusal_rate_table(df, group_col)
    wide = long_df.pivot(index=group_col, columns="pipeline_id")
    wide.columns = [f"{metric}__{PIPELINE_LABELS.get(pid, pid)}" for metric, pid in wide.columns]
    return wide.reset_index()


# Column -> display unit, used only to label the tables shown to the user; the
# underlying (unlabeled) column names are what later cells' lookups key off of.
METRIC_UNITS = {
    "n": "题", "n_standard": "题", "n_edge": "题",
    "context_precision": "0-1", "context_recall": "0-1", "context_relevancy": "0-1",
    "faithfulness": "0-1", "answer_relevancy": "0-1", "answer_correctness": "0-1",
    "refusal_correct_rate": "0-1", "wrong_refusal_type_rate": "0-1", "fabricated_rate": "0-1",
    "time_sec": "秒", "total_tokens": "tokens", "evidence_count": "条",
}


def _label_units(df: pd.DataFrame, units: dict[str, str] = METRIC_UNITS) -> pd.DataFrame:
    rename = {}
    for col in df.columns:
        if "__" not in col:
            continue
        metric, pid = col.rsplit("__", 1)
        if metric in units:
            rename[col] = f"{metric} ({units[metric]})__{pid}"
    return df.rename(columns=rename)


standard_df["type_category"] = standard_df["type"].map(TYPE_CATEGORY_LABELS)
edge_df["type_category"] = edge_df["type"].map(TYPE_CATEGORY_LABELS)
valid_df["type_category"] = valid_df["type"].map(TYPE_CATEGORY_LABELS)

# "多信息结合" is a subtotal over the 3 categories that all require combining more than
# one piece of retrieved information (as opposed to "单一事实", a single-fact lookup, and
# the edge category, which isn't a retrieval-quality comparison at all) -- all 3 are
# standard-type categories, so this subtotal never touches edge_df/refusal columns.
MULTI_INFO_LABEL = "多信息结合"
MULTI_INFO_CATEGORIES = ["多部分问题", "结构化页面", "跨页综合"]

_type_ragas = _pivot_by_pipeline(standard_df, "type_category", RAGAS_METRIC_COLUMNS)
_type_refusal = _pivot_refusal_by_pipeline(edge_df, "type_category")
_type_multi_info = _pivot_by_pipeline(
    standard_df[standard_df["type_category"].isin(MULTI_INFO_CATEGORIES)].assign(type_category=MULTI_INFO_LABEL),
    "type_category", RAGAS_METRIC_COLUMNS,
)

type_quality_table = (
    pd.concat([_type_ragas, _type_refusal, _type_multi_info], ignore_index=True)
    .set_index("type_category")
    .reindex(STANDARD_CATEGORY_ORDER + [MULTI_INFO_LABEL] + EDGE_CATEGORY_ORDER)
    .reset_index()
)
# total row: RAGAS averaged over all 30 standard rows + refusal rates over all 5 edge
# rows, merged into one row -- n counts the full 35 questions this row summarizes,
# not just the standard or edge half (unlike the per-category rows above, where n is
# unambiguous since a category is either pure-standard or pure-edge).
_type_n_total = _pivot_by_pipeline(valid_df.assign(type_category="total"), "type_category", [])
_type_ragas_total = _pivot_by_pipeline(standard_df.assign(type_category="total"), "type_category", RAGAS_METRIC_COLUMNS).drop(columns=["n__flatrag", "n__grag"])
_type_refusal_total = _pivot_refusal_by_pipeline(edge_df.assign(type_category="total"), "type_category").drop(columns=["n__flatrag", "n__grag"])
_type_total_row = _type_n_total.merge(_type_ragas_total, on="type_category").merge(_type_refusal_total, on="type_category")
type_quality_table = pd.concat([type_quality_table, _type_total_row], ignore_index=True)

type_cost_table = _pivot_by_pipeline(valid_df, "type_category", ["time_sec", "total_tokens", "evidence_count"])
_type_multi_info_cost = _pivot_by_pipeline(
    valid_df[valid_df["type_category"].isin(MULTI_INFO_CATEGORIES)].assign(type_category=MULTI_INFO_LABEL),
    "type_category", ["time_sec", "total_tokens", "evidence_count"],
)
type_cost_table = pd.concat([type_cost_table, _type_multi_info_cost], ignore_index=True)
type_cost_table = (
    type_cost_table.set_index("type_category")
    .reindex(STANDARD_CATEGORY_ORDER + [MULTI_INFO_LABEL] + EDGE_CATEGORY_ORDER)
    .reset_index()
)
_type_cost_total = _pivot_by_pipeline(valid_df.assign(type_category="total"), "type_category", ["time_sec", "total_tokens", "evidence_count"])
type_cost_table = pd.concat([type_cost_table, _type_cost_total], ignore_index=True)

display(_label_units(type_quality_table))
display(_label_units(type_cost_table))

,type_category,n (题)__flatrag,n (题)__grag,context_precision (0-1)__flatrag,context_precision (0-1)__grag,context_recall (0-1)__flatrag,context_recall (0-1)__grag,context_relevancy (0-1)__flatrag,context_relevancy (0-1)__grag,faithfulness (0-1)__flatrag,...,answer_relevancy (0-1)__flatrag,answer_relevancy (0-1)__grag,answer_correctness (0-1)__flatrag,answer_correctness (0-1)__grag,refusal_correct_rate (0-1)__flatrag,refusal_correct_rate (0-1)__grag,wrong_refusal_type_rate (0-1)__flatrag,wrong_refusal_type_rate (0-1)__grag,fabricated_rate (0-1)__flatrag,fabricated_rate (0-1)__grag
0,单一事实,10,10,0.57,1.00,0.70,1.00,0.85,1.0,0.86,...,0.74,0.96,0.48,0.74,NaN,NaN,NaN,NaN,NaN,NaN
1,多部分问题,10,10,0.80,0.59,0.75,0.85,0.95,1.0,0.91,...,0.85,0.95,0.74,0.84,NaN,NaN,NaN,NaN,NaN,NaN
2,结构化页面,5,5,0.80,0.60,1.00,1.00,1.00,1.0,0.87,...,0.90,0.96,0.74,0.77,NaN,NaN,NaN,NaN,NaN,NaN
3,跨页综合,5,5,0.52,0.58,1.00,0.80,1.00,1.0,0.93,...,0.86,0.96,0.84,0.56,NaN,NaN,NaN,NaN,NaN,NaN
4,多信息结合,20,20,0.73,0.59,0.88,0.88,0.98,1.0,0.90,...,0.86,0.96,0.76,0.75,NaN,NaN,NaN,NaN,NaN,NaN
5,无答案/边界问题,5,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.8,1.0,0.2,0.0,0.0,0.0
6,total,35,35,0.68,0.73,0.82,0.92,0.93,1.0,0.89,...,0.82,0.96,0.67,0.75,0.8,1.0,0.2,0.0,0.0,0.0


,type_category,n (题)__flatrag,n (题)__grag,time_sec (秒)__flatrag,time_sec (秒)__grag,total_tokens (tokens)__flatrag,total_tokens (tokens)__grag,evidence_count (条)__flatrag,evidence_count (条)__grag
0,单一事实,10,10,18.37,14.77,16016.70,8374.40,2.20,1.50
1,多部分问题,10,10,12.19,24.30,8850.20,13605.10,2.70,4.40
2,结构化页面,5,5,10.94,16.39,8068.40,9025.40,2.00,3.80
3,跨页综合,5,5,12.47,39.60,9355.20,24807.40,4.60,5.00
4,多信息结合,20,20,11.95,26.15,8781.00,15260.75,3.00,4.40
5,无答案/边界问题,5,5,20.10,44.45,20841.40,29714.00,6.60,2.20
6,total,35,35,14.95,25.51,12571.26,15357.97,3.29,3.26


把 `type_quality_table`/`type_cost_table`（含“多信息结合”子总计行和 `total` 行）换算成一张“GRAG v2 相对 FlatRAG 的增幅”表：RAGAS 6 项指标和拒答正确率是 0-1 量表，用 (grag − flatrag) × 100 算百分点（pp）差；延迟、token 用量不是 0-1 量表，改用相对变化的百分比 (grag − flatrag) / flatrag × 100；evidence 数用绝对差值 grag − flatrag。列名里直接标注口径单位（pp / % / Δ条）。三种口径都是正数 = GRAG v2 数值更大，不直接等于“更好”，还要结合指标本身的方向解读。

In [35]:
def _gain_table(table: pd.DataFrame, group_col: str, pp_metrics: list[str], pct_metrics: list[str], delta_metrics: list[str], delta_units: dict[str, str] | None = None) -> pd.DataFrame:
    delta_units = delta_units or {}
    rows = []
    for _, r in table.iterrows():
        row = {group_col: r[group_col]}
        for m in pp_metrics:
            flat_v, grag_v = r.get(f"{m}__flatrag"), r.get(f"{m}__grag")
            row[f"{m} (pp)"] = round((grag_v - flat_v) * 100, SCORE_ROUND) if pd.notna(flat_v) and pd.notna(grag_v) else math.nan
        for m in pct_metrics:
            flat_v, grag_v = r.get(f"{m}__flatrag"), r.get(f"{m}__grag")
            row[f"{m} (%)"] = round((grag_v - flat_v) / flat_v * 100, SCORE_ROUND) if pd.notna(flat_v) and pd.notna(grag_v) and flat_v else math.nan
        for m in delta_metrics:
            flat_v, grag_v = r.get(f"{m}__flatrag"), r.get(f"{m}__grag")
            unit = delta_units.get(m, "")
            row[f"{m} (Δ{unit})"] = round(grag_v - flat_v, SCORE_ROUND) if pd.notna(flat_v) and pd.notna(grag_v) else math.nan
        rows.append(row)
    return pd.DataFrame(rows)


_type_quality_gain = _gain_table(type_quality_table, "type_category", RAGAS_METRIC_COLUMNS + ["refusal_correct_rate"], [], [])
_type_cost_gain = _gain_table(type_cost_table, "type_category", [], ["time_sec", "total_tokens"], ["evidence_count"], delta_units={"evidence_count": "条"})
type_gain_table = _type_quality_gain.merge(_type_cost_gain, on="type_category")
type_gain_table

,type_category,context_precision (pp),context_recall (pp),context_relevancy (pp),faithfulness (pp),answer_relevancy (pp),answer_correctness (pp),refusal_correct_rate (pp),time_sec (%),total_tokens (%),evidence_count (Δ条)
0,单一事实,43.0,30.0,15.0,11.0,22.0,26.0,NaN,-19.60,-47.71,-0.70
1,多部分问题,-21.0,10.0,5.0,-12.0,10.0,10.0,NaN,99.34,53.73,1.70
2,结构化页面,-20.0,0.0,0.0,-14.0,6.0,3.0,NaN,49.82,11.86,1.80
3,跨页综合,6.0,-20.0,0.0,-14.0,10.0,-28.0,NaN,217.56,165.17,0.40
4,多信息结合,-14.0,0.0,2.0,-13.0,10.0,-1.0,NaN,118.83,73.79,1.40
5,无答案/边界问题,NaN,NaN,NaN,NaN,NaN,NaN,20.0,121.14,42.57,-4.40
6,total,5.0,10.0,7.0,-5.0,14.0,8.0,20.0,70.64,22.17,-0.03


总体而言，GRAG v2 的 `total` 行显示：召回端小幅全面领先——`context_recall` 0.82→0.92（+10pp）、`context_relevancy` 0.93→1.00（+7pp）、`context_precision` 0.68→0.73（+5pp）；回答端 `answer_relevancy` 0.82→0.96（+14pp）、`answer_correctness` 0.67→0.75（+8pp）同样提升，唯独 `faithfulness` 反而退步 5pp，是六项 RAGAS 指标里唯一走低的。代价是耗时 14.95s→25.51s（+70.64%）、token 12571→15358（+22.17%），evidence 数基本没变（3.29→3.26，Δ-0.03）。单看对用户最直接相关的 `answer_correctness`，+8pp 换 +70.64% 延迟，从纯效率角度不算划算——但这是全部题型平均后的结果，值不值得高度依赖具体场景，往下拆开看差异很大。

分类型看召回：GRAG 在"单一事实"上全面碾压（precision +43pp、recall +30pp、relevancy +15pp），是它检索最擅长的场景；但"多部分问题"（precision -21pp）和"结构化页面"（precision -20pp）上检索反而更不精准，说明这两类问题 GRAG 引入了更多噪音证据；"跨页综合"上 recall -20pp，是唯一召回为负的大类——这与 README 预期的"GRAG 应该更擅长跨页"正好相反。

分类型看回答质量：`answer_relevancy` 在所有大类都是正的（+5~22pp），生成阶段紧扣问题的能力稳定优于 FlatRAG；但 `answer_correctness` 在"跨页综合"上单独暴跌 -28pp，是本次评测里最差的单项结果——证据更多、也更贴题，最终答案正确性却明显下降，问题出在跨页证据的整合/推理环节而非检索环节。`faithfulness` 除"单一事实"外全线走低（-12~-14pp），和 precision 下降的类型高度重合：证据噪音变多后，生成阶段更容易说出证据不完全支持的内容。

归纳下来，GRAG v2 真正擅长两类场景：① 需要精确定位单一具体事实的问题（`single_fact`）——质量全面领先的同时还更快更省 token（-19.6% 耗时、-47.7% token），是唯一"又好又便宜"的场景；② 边界/拒答判断——`refusal_correct_rate` +20pp，`wrong_refusal_type_rate` 从 0.2 降到 0。它的短板集中在需要跨多个来源综合推理的问题上：证据更多但更杂，`answer_correctness`/`faithfulness` 反而承压，"跨页综合"是本次结果里唯一"更贵还更差"的类型。

## 第 4 段 — 按难度整体统计

复用第 2 段已经打分的 `valid_df`，按 `difficulty`（易/中/难）分组，采用和第 3 段完全一样的
质量表 / 成本表 / 增幅表三件套：标准 6 项 RAGAS 指标只对标准题型取均值，边界题不因为没有
RAGAS 分数就从统计里消失，用单独的"拒答正确率"列代表它们在该难度桶里的表现。

按 `difficulty` 分组，复用第 3 段定义的 `_pivot_by_pipeline`/`_pivot_refusal_by_pipeline`，输出两张表：质量指标表（标准题 6 项 RAGAS 指标均值 + 边界题拒答正确率/错误话术率/编造率，`easy`/`medium`/`hard` 各一行），最下面加一行覆盖全部 35 条问题的 `total` 行；成本指标表（两条 pipeline 在每个难度桶下的平均耗时、平均 token 用量、平均 evidence 数），同样加 `total` 行。两张表都按 pipeline 拆成 `flatrag`/`grag` 两列并在列名里标注单位，数值四舍五入到 2 位小数。

In [25]:
_DIFFICULTY_ORDER = ["easy", "medium", "hard"]

difficulty_std = valid_df[valid_df["type"].isin(STANDARD_TYPES)].copy()
difficulty_edge = valid_df[valid_df["type"].isin(EDGE_TYPES)].copy()


def _difficulty_quality_row(std_df: pd.DataFrame, edge_df_: pd.DataFrame) -> pd.DataFrame:
    ragas = _pivot_by_pipeline(std_df, "difficulty", RAGAS_METRIC_COLUMNS)
    ragas = ragas.rename(columns={"n__flatrag": "n_standard__flatrag", "n__grag": "n_standard__grag"})
    refusal = _pivot_refusal_by_pipeline(edge_df_, "difficulty")
    refusal = refusal.rename(columns={"n__flatrag": "n_edge__flatrag", "n__grag": "n_edge__grag"})
    return ragas.merge(refusal, on="difficulty", how="outer")


difficulty_quality_table = _difficulty_quality_row(difficulty_std, difficulty_edge)
difficulty_quality_table = difficulty_quality_table.set_index("difficulty").reindex(_DIFFICULTY_ORDER).reset_index()
_difficulty_total_row = _difficulty_quality_row(difficulty_std.assign(difficulty="total"), difficulty_edge.assign(difficulty="total"))
difficulty_quality_table = pd.concat([difficulty_quality_table, _difficulty_total_row], ignore_index=True)

difficulty_cost_table = _pivot_by_pipeline(valid_df, "difficulty", ["time_sec", "total_tokens", "evidence_count"])
difficulty_cost_table = difficulty_cost_table.set_index("difficulty").reindex(_DIFFICULTY_ORDER).reset_index()
_difficulty_cost_total = _pivot_by_pipeline(valid_df.assign(difficulty="total"), "difficulty", ["time_sec", "total_tokens", "evidence_count"])
difficulty_cost_table = pd.concat([difficulty_cost_table, _difficulty_cost_total], ignore_index=True)

display(_label_units(difficulty_quality_table))
display(_label_units(difficulty_cost_table))

,difficulty,n_standard (题)__flatrag,n_standard (题)__grag,context_precision (0-1)__flatrag,context_precision (0-1)__grag,context_recall (0-1)__flatrag,context_recall (0-1)__grag,context_relevancy (0-1)__flatrag,context_relevancy (0-1)__grag,faithfulness (0-1)__flatrag,...,answer_correctness (0-1)__flatrag,answer_correctness (0-1)__grag,n_edge (题)__flatrag,n_edge (题)__grag,refusal_correct_rate (0-1)__flatrag,refusal_correct_rate (0-1)__grag,wrong_refusal_type_rate (0-1)__flatrag,wrong_refusal_type_rate (0-1)__grag,fabricated_rate (0-1)__flatrag,fabricated_rate (0-1)__grag
0,easy,12,12,0.76,0.92,0.92,1.00,1.00,1.0,0.91,...,0.68,0.71,2,2,0.5,1.0,0.5,0.0,0.0,0.0
1,medium,12,12,0.68,0.69,0.75,0.92,0.92,1.0,0.89,...,0.68,0.79,2,2,1.0,1.0,0.0,0.0,0.0,0.0
2,hard,6,6,0.50,0.42,0.75,0.75,0.83,1.0,0.85,...,0.62,0.75,1,1,1.0,1.0,0.0,0.0,0.0,0.0
3,total,30,30,0.68,0.73,0.82,0.92,0.93,1.0,0.89,...,0.67,0.75,5,5,0.8,1.0,0.2,0.0,0.0,0.0


,difficulty,n (题)__flatrag,n (题)__grag,time_sec (秒)__flatrag,time_sec (秒)__grag,total_tokens (tokens)__flatrag,total_tokens (tokens)__grag,evidence_count (条)__flatrag,evidence_count (条)__grag
0,easy,14,14,13.68,25.78,10696.29,14611.93,3.43,2.21
1,medium,14,14,16.16,22.24,15282.43,12324.29,3.57,3.36
2,hard,7,7,15.04,31.53,10898.86,22917.43,2.43,5.14
3,total,35,35,14.95,25.51,12571.26,15357.97,3.29,3.26


把 `difficulty_quality_table`/`difficulty_cost_table`（含 `total` 行）换算成一张“GRAG v2 相对 FlatRAG 的增幅”表，口径和第 3 段的 `type_gain_table` 完全一致：RAGAS/拒答正确率用 pp 差，延迟/token 用相对百分比变化，evidence 数用绝对差值，列名标注 pp / % / Δ条。

In [26]:
_difficulty_quality_gain = _gain_table(difficulty_quality_table, "difficulty", RAGAS_METRIC_COLUMNS + ["refusal_correct_rate"], [], [])
_difficulty_cost_gain = _gain_table(difficulty_cost_table, "difficulty", [], ["time_sec", "total_tokens"], ["evidence_count"], delta_units={"evidence_count": "条"})
difficulty_gain_table = _difficulty_quality_gain.merge(_difficulty_cost_gain, on="difficulty")
difficulty_gain_table

,difficulty,context_precision (pp),context_recall (pp),context_relevancy (pp),faithfulness (pp),answer_relevancy (pp),answer_correctness (pp),refusal_correct_rate (pp),time_sec (%),total_tokens (%),evidence_count (Δ条)
0,easy,16.0,8.0,0.0,-6.0,2.0,3.0,50.0,88.45,36.61,-1.22
1,medium,1.0,17.0,8.0,0.0,20.0,11.0,0.0,37.62,-19.36,-0.21
2,hard,-8.0,0.0,17.0,-16.0,19.0,13.0,0.0,109.64,110.27,2.71
3,total,5.0,10.0,7.0,-5.0,14.0,8.0,20.0,70.64,22.17,-0.03


按难度拆开看，GRAG v2 的优势和代价都随难度上升而变化，但不是单调的。质量端：`answer_relevancy`/`answer_correctness` 在 medium（+20pp/+11pp）和 hard（+19pp/+13pp）两档的提升幅度都明显大于 easy（+2pp/+3pp），说明它的多轮 agent 能力确实更能应对复杂问题；但 `context_precision` 只在 hard 档转负（-8pp，三档里唯一负值），`faithfulness` 也在 hard 档跌得最狠（-16pp），说明难题上 GRAG 检索到更多证据（evidence 数 Δ+2.71，三档里唯一正值）但更不精准，生成阶段的幻觉风险也随之升高——是用更杂的证据换更好的最终答案。成本端：token 开销 easy 档 +36.61%、hard 档 +110.27%（翻倍以上），medium 档反而 -19.36%（三档里唯一变便宜的一档，配合它在质量上的高性价比提升，medium 是投入产出比最好的一档）；`refusal_correct_rate` 只在 easy 档有提升（+50pp，从 0.5→1.0），medium/hard 两档两条 pipeline 都已经是 1.0，没有进一步提升空间。整体上，难度越高，GRAG 的最终答案质量优势越明显，但伴随的是更高的检索噪音、更高的幻觉风险和成倍增长的成本，不是没有代价的提升。

## 第 5 段 — 商业性表现：各自最适合的场景和失败案例

仅在有标准 6 指标的 30 条范围内，按一个综合质量分（Faithfulness / Answer Relevancy /
Answer Correctness 的均值）排序，取两条 pipeline 各自的 top 7 / bottom 7；再单独看
分歧最大的问题（两条 pipeline 综合质量分差最大），以及边界题的拒答表现分歧。

给每条标准题计算一个综合质量分（Faithfulness / Answer Relevancy / Answer Correctness 的均值），并分别取两条 pipeline 各自综合质量分最高、最低的 7 条问题。

In [27]:
def _composite_quality(row: pd.Series) -> float:
    vals = [row.get(m) for m in ("faithfulness", "answer_relevancy", "answer_correctness")]
    vals = [v for v in vals if pd.notna(v)]
    return sum(vals) / len(vals) if vals else math.nan


standard_scored = standard_df.copy()
standard_scored["composite_quality"] = standard_scored.apply(_composite_quality, axis=1).round(SCORE_ROUND)

_TOP_BOTTOM_COLS = ["question_id", "question", "type", "difficulty", "composite_quality", "answer_correctness", "time_sec"]
_TOP_BOTTOM_ROUND_COLS = ["composite_quality", "answer_correctness", "time_sec"]


def _top_bottom(df: pd.DataFrame, pipeline_id: str, n: int = 7) -> tuple[pd.DataFrame, pd.DataFrame]:
    sub = df[df["pipeline_id"] == pipeline_id].dropna(subset=["composite_quality"]).sort_values(
        "composite_quality", ascending=False
    )
    sub = sub[_TOP_BOTTOM_COLS].copy()
    sub[_TOP_BOTTOM_ROUND_COLS] = sub[_TOP_BOTTOM_ROUND_COLS].round(SCORE_ROUND)
    return sub.head(n), sub.tail(n)


flat_top7, flat_bottom7 = _top_bottom(standard_scored, "pipeline_flatrag")
grag_top7, grag_bottom7 = _top_bottom(standard_scored, "pipeline_grag_v2")
print(f"FlatRAG: top {len(flat_top7)}, bottom {len(flat_bottom7)} (of {len(standard_scored[standard_scored.pipeline_id == 'pipeline_flatrag'])} scored)")
print(f"GRAG v2: top {len(grag_top7)}, bottom {len(grag_bottom7)} (of {len(standard_scored[standard_scored.pipeline_id == 'pipeline_grag_v2'])} scored)")

FlatRAG: top 7, bottom 7 (of 30 scored)
GRAG v2: top 7, bottom 7 (of 30 scored)


展示 FlatRAG 综合质量分最高的 7 条问题。

In [28]:
flat_top7

,question_id,question,type,difficulty,composite_quality,answer_correctness,time_sec
26,C12,Which sample elective courses are listed for t...,multi_fact,easy,0.99,0.97,11.14
48,F03,Which courses does Arnab Bose teach in the MS-...,multi_fact,medium,0.98,0.97,24.15
66,X04,Are the required core courses the same for the...,comparative,easy,0.98,0.95,11.66
64,X03,Which sample elective courses are listed for t...,cross_page,hard,0.97,0.92,9.65
24,C10,How are the six core courses sequenced across ...,multi_fact,hard,0.96,0.89,9.14
12,C02,Which core courses are scheduled in Quarter 1 ...,multi_fact,easy,0.95,0.85,16.22
58,T03,"How is the $1,500 non-refundable enrollment de...",single_fact,medium,0.95,0.95,11.93


展示 FlatRAG 综合质量分最低的 7 条问题。

In [29]:
flat_bottom7

,question_id,question,type,difficulty,composite_quality,answer_correctness,time_sec
16,C05,What's the difference between the 1-year 12-co...,comparative,medium,0.72,0.53,10.64
46,F01,Who is the director of the MS in Applied Data ...,single_fact,easy,0.69,0.19,11.89
32,C15,Which downtown Chicago buildings host In-Perso...,multi_fact,easy,0.68,0.71,10.29
54,T01,How much does the MS in Applied Data Science p...,single_fact,easy,0.66,0.28,15.20
2,A04,What application materials do I need to submit...,multi_fact,medium,0.33,0.16,10.48
18,C07,At what time do weekday live synchronous class...,single_fact,medium,0.33,0.19,34.82
62,T05,Which MS-ADS faculty member led AI at William ...,single_fact,hard,0.21,0.14,25.87


展示 GRAG v2 综合质量分最高的 7 条问题。

In [30]:
grag_top7

,question_id,question,type,difficulty,composite_quality,answer_correctness,time_sec
27,C12,Which sample elective courses are listed for t...,multi_fact,easy,0.99,0.98,16.83
47,F01,Who is the director of the MS in Applied Data ...,single_fact,easy,0.99,0.98,21.11
19,C07,At what time do weekday live synchronous class...,single_fact,medium,0.98,0.96,14.38
25,C10,How are the six core courses sequenced across ...,multi_fact,hard,0.98,0.98,30.26
53,S01,How does the age distribution differ between t...,table,hard,0.98,0.95,16.65
13,C02,Which core courses are scheduled in Quarter 1 ...,multi_fact,easy,0.98,0.98,14.40
59,T03,"How is the $1,500 non-refundable enrollment de...",single_fact,medium,0.96,0.99,15.42


展示 GRAG v2 综合质量分最低的 7 条问题。

In [31]:
grag_bottom7

,question_id,question,type,difficulty,composite_quality,answer_correctness,time_sec
31,C14,In which quarter of the sample 2-year full-tim...,single_fact,hard,0.75,0.46,20.98
51,F04,Which MS-ADS instructors have worked at Google?,multi_fact,hard,0.75,0.77,24.91
23,C09,Which noncredit courses are required or option...,accordion,easy,0.72,0.56,12.74
9,A07,Which standardized test scores are optional fo...,multi_fact,easy,0.70,0.57,50.24
49,F03,Which courses does Arnab Bose teach in the MS-...,multi_fact,medium,0.66,0.98,15.49
11,C01,Which six core courses are required in the In-...,accordion,easy,0.63,0.91,13.24
65,X03,Which sample elective courses are listed for t...,cross_page,hard,0.58,0.74,57.60


先说明一个限制：`divergence_table`（两条 pipeline 同题质量分差异表）和 `edge_pivot`（边界题拒答分歧表）这两个 cell 连同说明文字被一起删掉了，现在只剩 top7/bottom7 四张表，没法做"哪条 pipeline 在哪道具体题上明显更强"的精确配对分析，下面的场景判断只能基于间接观察，置信度比原来设计的分歧表弱——如果需要更硬的结论，建议把那两个 cell 加回来重新跑一次。

从现有的 top7/bottom7 能看到几个具体信号：FlatRAG 的 top7 里 4/7 是 multi_fact（C12/F03/C10/C02），说明它在"列出/拼接几个具体信息点"这类清单式问题上稳定；它的 bottom7 里反而 4/7 是 single_fact（F01/T01/C07/T05），且这几题的 `answer_correctness` 低至 0.14~0.28，像是对"总监是谁""学费多少"这类需要命中单一具体字段的问题检索失手。GRAG v2 的 top7 里恰好出现了 F01（composite 0.69→0.99）和 C07（0.33→0.98）——正是 FlatRAG 答得最差的两道题，说明 GRAG 在这类"点查询"上确实能修正 FlatRAG 的检索遗漏，和第 3 段"单一事实"大类的聚合结论吻合。反过来，X03（cross_page/hard）在 FlatRAG top7 排第 4（0.97），却掉进了 GRAG bottom7（0.58，`answer_correctness` 0.74），是一道具体的"倒退"案例，印证了第 3 段"跨页综合是 GRAG 最大短板"的结论。GRAG bottom7 里还有两道 accordion（C09/C01），且 C01 的 `answer_correctness` 其实有 0.91（内容基本正确），composite 分数低更像是被 `faithfulness`/`answer_relevancy` 拉低，而不是真答错了——提示 accordion 这类折叠内容对 GRAG 来说，问题可能出在措辞/引用方式上，而不是检索到错误信息。

综合来看，比较适合 FlatRAG 的场景是清单式、需要拼接多个信息点的 multi_fact 问题；比较适合 GRAG v2 的场景是需要精确命中单一字段的 single_fact 问题；GRAG v2 明显偏弱、需要谨慎使用的场景是跨页综合（cross_page/comparative）和 accordion 类结构化内容。